# Demo N-gram Language Model với WikiText-2

Notebook này minh họa cách xây dựng Unigram, Bigram và Trigram language model trên dataset WikiText-2.


# Import thư viện

In [1]:
import math
import random
import numpy as np
from collections import defaultdict, Counter

from datasets import load_dataset

# Load corpus
Data sử dụng: WikiText-2 raw corpus từ Hugging Face để xây dựng N-gram language model.

WikiText-2 là dataset văn bản từ Wikipedia, thường được dùng cho bài toán language modeling.


In [2]:
from datasets import load_dataset

ds = load_dataset("wikitext", "wikitext-2-raw-v1")
text = " ".join(ds["train"]["text"]).lower()

# Loại bỏ các dòng rỗng quá nhiều để corpus sạch hơn
text = " ".join(text.split())

print("Dataset:", ds)
print("Corpus length:", len(text))
print(text[:500])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Dataset: DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})
Corpus length: 10845407
= valkyria chronicles iii = senjō no valkyria 3 : unrecorded chronicles ( japanese : 戦場のヴァルキュリア3 , lit . valkyria of the battlefield 3 ) , commonly referred to as valkyria chronicles iii outside japan , is a tactical role @-@ playing video game developed by sega and media.vision for the playstation portable . released in january 2011 in japan , it is the third game in the valkyria series . employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs paral


## Tokenize

Chuyển corpus thành danh sách các tokens (words) để phục vụ việc xây dựng N-gram model.



In [3]:
tokens = text.split()

print("Total tokens:", len(tokens))
print(tokens[:20])

Total tokens: 2051910
['=', 'valkyria', 'chronicles', 'iii', '=', 'senjō', 'no', 'valkyria', '3', ':', 'unrecorded', 'chronicles', '(', 'japanese', ':', '戦場のヴァルキュリア3', ',', 'lit', '.', 'valkyria']


# Build Unigram Model

In [4]:
unigram_counts = Counter(tokens)

total_words = len(tokens)

unigram_prob = {}

for word,count in unigram_counts.items():

    unigram_prob[word] = count / total_words

print("Example probabilities")

for w in list(unigram_prob.keys())[:10]:
    print(w, unigram_prob[w])

Example probabilities
= 0.014410963443815763
valkyria 2.6316943725602e-05
chronicles 2.582959291586863e-05
iii 0.00011257803704840855
senjō 2.436754048666852e-06
no 0.0006808290811975184
3 0.0007802486463831259
: 0.0018548571818452076
unrecorded 1.9494032389334816e-06
( 0.005844310910322578


# Build Bigram Model
Xây dựng Bigram model để ước lượng xác suất của một từ dựa trên từ đứng trước nó.

$$\text{Bigram} = P(w_2 | w_1)$$


In [5]:
bigram_counts = defaultdict(Counter)

for i in range(len(tokens)-1):

    w1 = tokens[i]
    w2 = tokens[i+1]

    bigram_counts[w1][w2] += 1

cnt = 0
print("Example bigram counts")
print(bigram_counts["the"].most_common(5))

Example bigram counts
[('first', 2223), ('song', 1147), ('game', 950), ('same', 908), ('united', 875)]


In [6]:
bigram_prob = {}

for w1 in bigram_counts:

    total = sum(bigram_counts[w1].values())

    bigram_prob[w1] = {}

    for w2 in bigram_counts[w1]:

        bigram_prob[w1][w2] = bigram_counts[w1][w2] / total

In [7]:
print("Example bigram probabilities")

for w2,p in list(bigram_prob["the"].items())[:5]:
    print("P(",w2,"| the ) =",p)

Example bigram probabilities
P( battlefield | the ) = 0.00023706105469227946
P( playstation | the ) = 0.0005200048941637098
P( third | the ) = 0.0024700232472776216
P( valkyria | the ) = 5.352991557567601e-05
P( same | the ) = 0.00694359476324483


# Build Trigram Model
Xây dựng Bigram model để ước lượng xác suất của một từ dựa trên từ đứng trước nó.

$$\text{Trigram} = P(w_3 | w_1, w_2)$$

In [8]:
trigram_counts = defaultdict(Counter)

for i in range(len(tokens)-2):
    w1 = tokens[i]
    w2 = tokens[i+1]
    w3 = tokens[i+2]
    trigram_counts[(w1,w2)][w3] += 1
print("Example trigram")
key = list(trigram_counts.keys())[0]
print(key, trigram_counts[key].most_common(5))

Example trigram
('=', 'valkyria') [('chronicles', 2)]


In [9]:
trigram_prob = {}

for key in trigram_counts:
    total = sum(trigram_counts[key].values())
    trigram_prob[key] = {}
    for w3 in trigram_counts[key]:
        trigram_prob[key][w3] = trigram_counts[key][w3] / total

In [10]:
key = list(trigram_prob.keys())[0]

sorted(trigram_prob[key].items(), key=lambda x: x[1], reverse=True)[:5]

[('chronicles', 1.0)]

# Sentence Log Probability
Xây dựng Trigram model để dự đoán xác suất của một từ dựa trên hai từ trước đó.

In [11]:
def sentence_log_probability(sentence):
    words = sentence.lower().split()
    log_prob = 0
    for i in range(len(words)-1):
        w1 = words[i]
        w2 = words[i+1]
        if w1 in bigram_prob and w2 in bigram_prob[w1]:
            log_prob += math.log(bigram_prob[w1][w2])
        else:
            log_prob += math.log(1e-6)
    return log_prob

In [12]:
sentence_log_probability("the king is dead")

-20.29666412120219

# Perplexity Function

Tính perplexity để đánh giá mức độ dự đoán của language model đối với một câu hoặc tập dữ liệu.

$$
PP(W) =
\exp \left(
  - \frac{1}{N}  
  \sum_{i=1}^{N} \log P(w_i | w_{i-1})
\right)
$$


In [13]:
def perplexity(sentence):
    words = sentence.lower().split()
    N = len(words)
    log_prob = sentence_log_probability(sentence)
    return math.exp(-log_prob / N)

In [14]:
sentences = [
    "the city is located in the northern part of the country",
    "the album was released in the united states in 1999",
    "the species is found in tropical and subtropical regions",
    "the film received positive reviews from critics",
    "the river flows through the central part of the island",
    "the population was estimated at about 20,000 people",
    "the church was built in the late nineteenth century",
    "the book was first published in the united kingdom",
    "the team played its home games at the stadium",
    "the area is known for its historical buildings"
]

for s in sentences:
    pp = perplexity(s)
    print(f"{s}  ->  perplexity: {pp:.2f}")

the city is located in the northern part of the country  ->  perplexity: 23.14
the album was released in the united states in 1999  ->  perplexity: 19.64
the species is found in tropical and subtropical regions  ->  perplexity: 591.42
the film received positive reviews from critics  ->  perplexity: 18.99
the river flows through the central part of the island  ->  perplexity: 31.73
the population was estimated at about 20,000 people  ->  perplexity: 581.97
the church was built in the late nineteenth century  ->  perplexity: 24.35
the book was first published in the united kingdom  ->  perplexity: 23.49
the team played its home games at the stadium  ->  perplexity: 123.97
the area is known for its historical buildings  ->  perplexity: 60.16


# Generate Text Using Bigram

Sinh văn bản mới bằng cách lấy mẫu từ Bigram probability distribution đã học.

In [15]:
def generate_bigram(start_word, length=20):

    word = start_word
    sentence = [word]

    for _ in range(length):

        if word not in bigram_counts:
            break

        next_words = list(bigram_counts[word].keys())
        weights = list(bigram_counts[word].values())

        word = random.choices(next_words, weights)[0]

        sentence.append(word)

    return " ".join(sentence)

In [20]:
generate_bigram("the")

"the southern somalia was found in command , and private intrigue and thea 's first performance , camacho attempted negotiations with"